Reading up on building a basic RAG system, using different embedding strategies. This helps us to know which embedding strategy will work best for the document types that we have

https://pub.towardsai.net/mastering-rag-choosing-the-right-vector-embedding-model-for-your-rag-application-bbb57517890e

Testing out pdf parsing using PyMuPDF

In [1]:
# !pip install pymupdf.layout
# !pip install PyMuPDF
# !pip install pymupdf4llm
# !pip install matplotlib
# # !pip install unstructured
# !pip install langchain-huggingface sentence-transformers
# !pip install --upgrade torch transformers accelerate
# !pip install --upgrade torchvision
# !pip install langchain_qdrant
!pip install --upgrade qdrant-client langchain-qdrant

Trying to test pdf parser

In [2]:
import fitz  # PyMuPDF
import pymupdf.layout
import pymupdf4llm
doc = fitz.open("../../info/MyNewdataset.pdf")

json = pymupdf4llm.to_json(doc)
print(json)


{"filename": "../../info/MyNewdataset.pdf", "page_count": 4, "toc": [], "pages": [{"page_number": 1, "width": 841.9199829101562, "height": 1191.1199951171875, "boxes": [{"x0": 25.51171875, "y0": 15.929421424865723, "x1": 85.9620361328125, "y1": 22.882104873657227, "boxclass": "page-header", "image": null, "table": null, "textlines": [{"bbox": [25.51171875, 15.929421424865723, 85.9620361328125, 22.882104873657227], "spans": [{"size": 7.994999885559082, "flags": 0, "bidi": 0, "char_flags": 16, "font": "ArialMT", "color": 0, "alpha": 255, "ascender": 0.800000011920929, "descender": -0.20000000298023224, "text": "3/26/26, 8:52 AM", "origin": [25.51171875, 21.75], "bbox": [25.51171875, 15.929421424865723, 85.9620361328125, 22.882104873657227], "line": 0, "block": 31, "dir": [1.0, 0.0]}]}]}, {"x0": 440.30859375, "y0": 16.027015686035156, "x1": 493.18695068359375, "y1": 23.432540893554688, "boxclass": "page-header", "image": null, "table": null, "textlines": [{"bbox": [440.30859375, 16.027015

Testing out unstructured pdf parser

mac: brew install poppler tesseract

In [3]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os
from unstructured.partition.auto import partition

In [20]:
print(os.getcwd())
path = "../data/train/pdfs_train/0b85477387a9d0cc33fca0f4becaa0e5.pdf"
os.path.exists(path)

/Users/bebs/Documents/School/DAT 560/DAT560project/src/notebooks


True

In [21]:
# !pip install "unstructured[pdf]"
blocks = partition(filename=path)

In [22]:
for block in blocks:
    print(f"{block.category}: {block.text}")


Header: NMR&D News
Title: Volume IV, Issue 12
Title: In this issue…
NarrativeText: 2 CO‟s Messages 3 Building Afghan Medical Capacity 4 USNS Mercy Pacific Partnership 5 DoD Bone Marrow Donor Program 6 ID Joint Planning Group 7 Capacity Building in Liberia 8 Kazakh Scientists Train at NMRC 9 Patient Condition Occurrence Tool 10 Combat Casualty Research Team 11 Accelerating Technology Transfer 12 NMRC Hosts Dining Out 13 Villasante Speaks at Notre Dame 14 Keane-Myers Speaks at Hopkins 14 Cub Scouts Learn Flag Etiquette 15 NMRC High School Outreach NMRC Officers Teach Science 15 2012 Combined Federal Campaign 16 16 Ombudsman‟s Note
NarrativeText: NMR&D News is an authorized publica- tion of the Naval Medical Research Center, 503 Robert Grant Avenue, Silver Spring, MD 20910. NMR&D News is published monthly by the NMRC Public Affairs Office, 301-319-9378 or svc.pao.nmrc@med.navy.mil .
NarrativeText: Commanding Officer Capt. John W. Sanders
Title: Executive Officer Capt. Elizabeth Montcalm-S

Test Embedding using KaLM

In [7]:
from langchain_huggingface import HuggingFaceEmbeddings
import sentence_transformers
from transformers import PreTrainedModel


# Define the KaLM model from Hugging Face
model_name = "KaLM-Embedding/KaLM-embedding-multilingual-mini-instruct-v2.5"
model_kwargs = {'device': 'cpu'} # Change to 'cuda' for GPU
encode_kwargs = {'normalize_embeddings': True}

# Initialize LangChain embedding class
kalm_embeddings = HuggingFaceEmbeddings(
    model_name=model_name,
    model_kwargs=model_kwargs,
    encode_kwargs=encode_kwargs
)

# Use the model to embed text
text = "This is a test document."
embedding = kalm_embeddings.embed_query(text)
print(embedding) # Outputs the embedding vector
print(len(embedding)) # Outputs the dimension of the embedding


/opt/homebrew/Caskroom/miniconda/base/envs/dat540/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 290/290 [00:00<00:00, 5284.24it/s]


[-0.0383729450404644, -0.026311155408620834, 0.01223372109234333, -0.019534660503268242, -0.07257401943206787, 0.02025051787495613, -0.028576504439115524, -0.0037212723400443792, -0.0028800664003938437, 0.03383132442831993, -0.04248383268713951, 0.06163700670003891, 0.0648241937160492, -0.022169407457113266, -0.02616947330534458, -0.012430868111550808, -0.024700095877051353, 0.09194932132959366, -0.023991065099835396, -0.023246068507432938, -0.0024326490238308907, 0.07014287263154984, -0.026182491332292557, 0.09707958996295929, -0.08901002258062363, -0.03617897629737854, 0.07076911628246307, -0.08108171075582504, -0.019471367821097374, 0.03488931059837341, 0.014957066625356674, 0.02167549729347229, -0.06686847656965256, -0.0413714163005352, 0.016199979931116104, 0.07999475300312042, -0.0491577573120594, 0.020232416689395905, 0.022344456985592842, 0.011296095326542854, -0.004685695748776197, -0.047681257128715515, 0.01805032789707184, 0.04005235433578491, -0.026294870302081108, 0.025115

Test storing embedding on vector db

In [12]:

from unstructured.partition.pdf import partition_pdf
from qdrant_client import QdrantClient
from qdrant_client.models import Distance, VectorParams, PointStruct
from sentence_transformers import SentenceTransformer
import uuid

COLLECTION_NAME = "mmdocir_baseline"

# Connect to the local Qdrant database you started with Docker
print("Connecting to Qdrant...")
client = QdrantClient(url="http://localhost:6333")

# Delete the existing incorrect collection if it exists
if client.collection_exists(collection_name=COLLECTION_NAME):
    client.delete_collection(collection_name=COLLECTION_NAME)
    print(f"Deleted old collection: {COLLECTION_NAME}")

# Create a fresh storage collection with the correct size (896)
client.create_collection(
    collection_name=COLLECTION_NAME,
    vectors_config=VectorParams(size=896, distance=Distance.COSINE),
)


print("Translating text to vectors and uploading to Qdrant...")

points_to_upload = []

# Loop through each extracted block of text
for i, block in enumerate(blocks):
    # Skip empty blocks
    if not block.text.strip():
        continue
        
    # Turn the text into a math vector using our LangChain embedding model
    vector = kalm_embeddings.embed_query(block.text) 
    
    # Create a unique ID for this chunk
    point_id = str(uuid.uuid4()) 
    
    # Package it up for Qdrant: ID + Math Vector + Original Text (Payload)
    point = PointStruct(
        id=point_id,
        vector=vector,
        payload={
            "source": path,
            "text": block.text,
            "category": block.category,
            "chunk_index": i
        }
    )
    points_to_upload.append(point)

# Send the massive batch of data to Qdrant all at once
if points_to_upload:
    client.upsert(
        collection_name=COLLECTION_NAME,
        points=points_to_upload
    )

print("✅ Success! PDF text is now indexed and ready for retrieval.")
# ...existing code...

Connecting to Qdrant...
Deleted old collection: mmdocir_baseline
Translating text to vectors and uploading to Qdrant...
✅ Success! PDF text is now indexed and ready for retrieval.


simple test query with vector db

In [16]:
#embed user query
query = "what are the stages in the annotation pipeline?"
query_vector = kalm_embeddings.embed_query(query)

# Perform a similarity search in Qdrant using the query vector
search_results = client.query_points(
    collection_name=COLLECTION_NAME,
    query=query_vector,
    limit=5  # Number of similar items to retrieve
)

# Print the search results
print( search_results)
# for result in search_results:
#     print(f"ID: {result.id}, Score: {result.score}")
#     print(f"Text: {result.payload['text']}\n")

points=[ScoredPoint(id='9fb73b39-0944-45b9-9bd5-9704c47a73aa', version=1, score=0.74691427, payload={'source': '../../info/MyNewdataset.pdf', 'text': 'The annotation pipeline of includes three stages. 1. (1) Data collection: We collect 364 long documents and 2,193 QA pairs from MMLongBench-Doc and DocBench, selecting datasets that include accessible original documents, diverse domains (e.g., academic, legal, \x00nancial), and rich multi-modal content such as text, \x00gures, tables, and layouts. The average document length exceeds 65 pages, ensuring the benchmark re\x00ects real-world document complexity. (2) Question Filtering & Adaptation:', 'category': 'NarrativeText', 'chunk_index': 34}, vector=None, shard_key=None, order_value=None), ScoredPoint(id='6f0d6abb-bc1d-4ff7-986b-2d4a72049eca', version=1, score=0.58377266, payload={'source': '../../info/MyNewdataset.pdf', 'text': 'To ensure alignment with retrieval objectives, we \x00lter out questions that are unsuitable for document-ba

DOCKER COMPOSE FOR QDrant

services:
  qdrant:
    image: qdrant/qdrant:latest
    container_name: qdrant_db
    ports:
      - "6333:6333"
    volumes:
      - ./qdrant_data:/qdrant/storage
    restart: always

Testing out kiran's chunking class, made some update will discuss with him tomorrow

Also added enhanced_hierarchical for cross page

In [16]:
import json
from typing import List, Dict, Any

class Chunking:
    HEADING_CATEGORIES = {"Title", "Header"}

    def __init__(self, blocks: List):
        self.blocks = blocks

    @staticmethod
    def _make_chunk(texts: List[str], extra_meta: Dict[str, Any] = None) -> Dict[str, Any]:
        """Helper function to store char_len which can be useful downstream"""
        chunk_text = " ".join(texts)
        chunk = {"text": chunk_text, "char_len": len(chunk_text)}
        if extra_meta:
            chunk.update(extra_meta)
        return chunk

    def fixed_size(self, max_chars: int = 1000) -> List[Dict[str, Any]]:
        """Concatenate blocks until max_chars is exceeded, then start a new chunk."""

        current_texts, chunks = [], []
        text_len, blocks_count = 0, len(self.blocks)
        for i, block in enumerate(self.blocks):
            text = block.text.strip()
            if not text:
                continue

            text_len += len(text)
            current_texts.append(text)
            if text_len > max_chars or i == blocks_count - 1:
                chunks.append(Chunking._make_chunk(current_texts))
                current_texts = []
                text_len = 0
        return chunks

    def sliding_window(
        self, window_charts: int = 1000, overlap_chars: int = 200
    ) -> List[Dict[str, Any]]:
        """
        Fixed-size windows with overlap so context at chunk boundaries
        is not lost. Good for dense text documents.
        """
        current_texts, chunks = [], []
        text_len, blocks_count = 0, len(self.blocks)
        for i, block in enumerate(self.blocks):
            text = block.text.strip()
            if not text:
                continue

            text_len += len(text)
            current_texts.append(text)
            if text_len > window_charts or i == blocks_count - 1:
                if chunks:
                    prev_text = chunks[-1]["text"][-overlap_chars:]  # use the actual text from the last chunk
                    current_texts = [prev_text] + current_texts
                else:
                    prev_text = ""
                
                current_texts.append(prev_text)
                chunks.append(Chunking._make_chunk(current_texts))
                current_texts = []
                text_len = overlap_chars
        
        return chunks

    def semantic(self) -> List[Dict[str, Any]]:
        """
        Group blocks into sections separated by title/header elements.
        Each heading starts a new chunk. This keeps the documents natural
        semantic boundaries intact.
        
        Consecutive headers without body text are merged forward so they
        attach to the next chunk that has actual content.
        """

        current_texts, chunks = [], []
        chunk_contains_text = False
        
        for block in self.blocks:
            text = block.text.strip()
            if not text:
                continue
            
            if block.category in self.HEADING_CATEGORIES:
                # Only flush if the current chunk has body text
                if chunk_contains_text:
                    chunks.append(Chunking._make_chunk(current_texts))
                    current_texts = []
                    chunk_contains_text = False
                
                current_texts.append(text)

            else:
                current_texts.append(text)
                chunk_contains_text = True
        
        # Add remaining - but if it's only headers, merge into the last chunk
        if current_texts:
            if chunk_contains_text or not chunks:
                chunks.append(Chunking._make_chunk(current_texts))
            else:
                # Trailing headers only - append to previous chunk
                prev = chunks[-1]
                combined = prev["text"] + " " + " ".join(current_texts)
                chunks[-1] = {"text": combined, "char_len": len(combined)}
        
        return chunks
    
    def hierarchical(self) -> List[Dict[str, Any]]:
        """
        Two-level hierarchy:
          - Level 1 (parent): full page text  (coarse retrieval)
          - Level 2 (child):  per-section text within each page  (fine retrieval)

        Returns a flat list but each chunk's metadata includes
        'level' ('page' or 'section') and 'page_number'.
        """
        # Group blocks by page number
        pages: dict[int, list] = {}
        for block in self.blocks:
            page = getattr(block.metadata, "page_number", None) or 0
            pages.setdefault(page, []).append(block)

        hierarchical_pages: list[dict] = []

        for page_num in sorted(pages):
            page_blocks = pages[page_num]
            page_texts = [b.text.strip() for b in page_blocks if b.text.strip()]

            if not page_texts:
                continue

            # Parent chunk: whole page
            page_chunk = Chunking._make_chunk(
                page_texts,
                extra_meta={"level": "page", "page_number": page_num},
            )


            # Child chunks: split by headings within the page
            page_sections: list[dict] = []
            section_texts: list[str] = []
            has_body = False
            for block in page_blocks:
                text = block.text.strip()
                if not text:
                    continue
                if block.category in self.HEADING_CATEGORIES:
                    if has_body and section_texts:
                        page_sections.append(
                            Chunking._make_chunk(
                                section_texts,
                                extra_meta={"level": "section", "page_number": page_num},
                            )
                        )
                        section_texts = []
                        has_body = False
                    section_texts.append(text)
                else:
                    section_texts.append(text)
                    has_body = True

            if section_texts:
                if has_body or not page_sections:
                    page_sections.append(
                        Chunking._make_chunk(
                            section_texts,
                            extra_meta={"level": "section", "page_number": page_num},
                        )
                    )
                else:
                    # Trailing headers - merge into last section chunk for this page
                    prev = page_sections[-1]
                    combined = prev["text"] + " " + " ".join(section_texts)
                    page_sections[-1] = {**prev, "text": combined, "char_len": len(combined)}

            # 3. Nest children inside the parent and append to final list
            page_chunk["chunks"] = page_sections
            hierarchical_pages.append(page_chunk)

        return hierarchical_pages

    def enhanced_hierarchical(self) -> List[Dict[str, Any]]:
        """
        Optimized Two-level hierarchy (Cross-Page):
        - Level 1 (parent): Full Section (Starts at a Heading, ends at the next Heading)
        - Level 2 (child): Individual paragraphs/blocks within that section.
        
        This prevents context loss when a logical section spills across a physical page break.
        Consecutive headers without body text are merged together.
        """
        hierarchical_sections: List[Dict[str, Any]] = []
        
        # State tracking for the current section
        current_parent_title = "Document Start" # Default if no header exists at the very beginning
        current_children: List[Dict[str, Any]] = []
        current_parent_texts: List[str] = []
        current_pages = set() # Using a set to track all pages this section spans
        has_body = False # Tracks if the current section has actual content
        
        for block in self.blocks:
            text = block.text.strip()
            if not text:
                continue
                
            # Get page number safely, default to 0 if missing
            page_num = getattr(block.metadata, "page_number", None) or 0
            is_heading = block.category in getattr(self, "HEADING_CATEGORIES", [])
            
            if is_heading:
                # 1. Flush ONLY if the current section actually has body text
                if has_body:
                    parent_chunk = self._make_chunk(
                        current_parent_texts,
                        extra_meta={
                            "level": "section", 
                            "title": current_parent_title,
                            "pages_spanned": sorted(list(current_pages))
                        }
                    )
                    parent_chunk["chunks"] = current_children
                    hierarchical_sections.append(parent_chunk)
                    
                    # Reset the state for the new section
                    current_parent_title = text
                    current_children = []
                    current_parent_texts = [text]
                    current_pages = {page_num}
                    has_body = False
                else:
                    # 2. Consecutive headings: Merge them together
                    if current_parent_title == "Document Start" and not current_parent_texts:
                        current_parent_title = text
                    else:
                        current_parent_title += f" {text}"
                        
                    current_parent_texts.append(text)
                    current_pages.add(page_num)
                
            else:
                # 3. We are inside a section body. Append text and track the page.
                has_body = True
                current_parent_texts.append(text)
                current_pages.add(page_num)
                
                # If this is the first child in the section, prepend the title to its text
                child_text = text
                if len(current_children) == 0 and current_parent_title != "Document Start":
                    child_text = f"{current_parent_title}\n{text}"
                
                # Create a child chunk for this specific paragraph/block
                current_children.append(
                    self._make_chunk(
                        [child_text],
                        extra_meta={
                            "level": "paragraph", 
                            "page_number": page_num,
                            "parent_title": current_parent_title
                        }
                    )
                )
                
        # 4. Flush the final section after the loop finishes
        if current_children or current_parent_texts:
            parent_chunk = self._make_chunk(
                current_parent_texts,
                extra_meta={
                    "level": "section", 
                    "title": current_parent_title,
                    "pages_spanned": sorted(list(current_pages))
                }
            )
            parent_chunk["chunks"] = current_children
            hierarchical_sections.append(parent_chunk)
            
        return hierarchical_sections

In [35]:
chunker = Chunking(blocks)
print(json.dumps(chunker.hierarchical(), indent=4))

[
    {
        "text": "NMR&D News Volume IV, Issue 12 In this issue\u2026 2 CO\u201fs Messages 3 Building Afghan Medical Capacity 4 USNS Mercy Pacific Partnership 5 DoD Bone Marrow Donor Program 6 ID Joint Planning Group 7 Capacity Building in Liberia 8 Kazakh Scientists Train at NMRC 9 Patient Condition Occurrence Tool 10 Combat Casualty Research Team 11 Accelerating Technology Transfer 12 NMRC Hosts Dining Out 13 Villasante Speaks at Notre Dame 14 Keane-Myers Speaks at Hopkins 14 Cub Scouts Learn Flag Etiquette 15 NMRC High School Outreach NMRC Officers Teach Science 15 2012 Combined Federal Campaign 16 16 Ombudsman\u201fs Note NMR&D News is an authorized publica- tion of the Naval Medical Research Center, 503 Robert Grant Avenue, Silver Spring, MD 20910. NMR&D News is published monthly by the NMRC Public Affairs Office, 301-319-9378 or svc.pao.nmrc@med.navy.mil . Commanding Officer Capt. John W. Sanders Executive Officer Capt. Elizabeth Montcalm-Smith Director for Administration L

In [17]:
chunker = Chunking(blocks)
print(json.dumps(chunker.enhanced_hierarchical(), indent=4))

[
    {
        "text": "NMR&D News Volume IV, Issue 12 In this issue\u2026 2 CO\u201fs Messages 3 Building Afghan Medical Capacity 4 USNS Mercy Pacific Partnership 5 DoD Bone Marrow Donor Program 6 ID Joint Planning Group 7 Capacity Building in Liberia 8 Kazakh Scientists Train at NMRC 9 Patient Condition Occurrence Tool 10 Combat Casualty Research Team 11 Accelerating Technology Transfer 12 NMRC Hosts Dining Out 13 Villasante Speaks at Notre Dame 14 Keane-Myers Speaks at Hopkins 14 Cub Scouts Learn Flag Etiquette 15 NMRC High School Outreach NMRC Officers Teach Science 15 2012 Combined Federal Campaign 16 16 Ombudsman\u201fs Note NMR&D News is an authorized publica- tion of the Naval Medical Research Center, 503 Robert Grant Avenue, Silver Spring, MD 20910. NMR&D News is published monthly by the NMRC Public Affairs Office, 301-319-9378 or svc.pao.nmrc@med.navy.mil . Commanding Officer Capt. John W. Sanders",
        "char_len": 881,
        "level": "section",
        "title": "NMR&D